# Sonata + VoxelGroupClassifier: Full Pipeline on Colab
## Entropy-Guided Adaptive Masking for 3D Scene Pretraining

**Requirements**: Colab with T4/V100/A100 GPU (Pro/Pro+ recommended).

**Steps**: Clone → Install → Convert Data → Train → Monitor

> ⚠️ After installing spconv, Colab will ask you to **restart the runtime**.
> Click "Restart Runtime", then continue from the same cell.

In [ ]:
# ═══════════════════════════════════════════════
# CONFIGURATION — change these before running
# ═══════════════════════════════════════════════
EPOCHS = 10          # training epochs
MASKING_MODE = "cosine"  # "random" | "cosine" | "vgc"
GPU_COUNT = 1        # number of GPUs to use

In [ ]:
# ═══════════════════════════════════════
# CELL 1 — Install Environment (5 min)
# ═══════════════════════════════════════
import sys, os, subprocess

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Clone repo
!rm -rf voxel_group_classifier
!git clone --depth 1 https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git
%cd voxel_group_classifier

# Install base deps
!pip install -q h5py tensorboard timm torch-scatter ninja

# Install spconv (CUDA 12.x prebuilt)
!pip install -q spconv-cu120 2>/dev/null || pip install -q spconv-cu121

# Compile CUDA extensions
%cd libs/pointops && python setup.py install --quiet 2>&1 | tail -3
%cd ../pointops2 && python setup.py install --quiet 2>&1 | tail -3
%cd ../pointgroup_ops && python setup.py install --quiet 2>&1 | tail -3
%cd ../..

# Verify
import torch, spconv
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"spconv {spconv.__version__}")
# Test spconv works
test_conv = spconv.SubMConv3d(3, 3, 3, bias=False).cuda()
x = torch.rand(1, 3, 64, 64, 64).cuda()
print(f"spconv test: {test_conv(x).shape}  ✅")

In [ ]:
# ═══════════════════════════════════════
# CELL 2 — Data: HDF5 → Pointcept (3 min)
# ═══════════════════════════════════════
import numpy as np, h5py, glob

# Download HDF5 from HuggingFace mirror (~1.7 GB)
HDF5_URL = "https://huggingface.co/datasets/cminst/S3DIS/resolve/main/indoor3d_sem_seg_hdf5_data.zip"
HDF5_DIR = "./indoor3d_sem_seg_hdf5_data"
OUT_ROOT = "./data/s3dis"

if not os.path.isdir(HDF5_DIR):
    print(f"Downloading S3DIS HDF5 from HuggingFace (~1.7 GB)...")
    import urllib.request, zipfile
    urllib.request.urlretrieve(HDF5_URL, "/tmp/s3dis.zip")
    with zipfile.ZipFile("/tmp/s3dis.zip") as zf:
        zf.extractall(HDF5_DIR)
    print("Download complete.")
else:
    print("Already downloaded.")

# Convert HDF5 → Pointcept format (merge blocks per HDF5 file into one room)
h5_files = sorted(glob.glob(f"{HDF5_DIR}/*/*.h5"))
print(f"Converting {len(h5_files)} HDF5 files to Pointcept format...")

written = 0
for h5_path in h5_files:
    fname = os.path.basename(h5_path).replace(".h5", "")
    area_idx = int(fname.split("_")[-1])
    area = f"Area_{area_idx}"

    with h5py.File(h5_path, "r") as f:
        data = f["data"][:]
        labels = f["label"][:]

    xyz = data[..., :3].reshape(-1, 3).astype(np.float32)
    rgb = data[..., 3:6].reshape(-1, 3).astype(np.uint8)
    seg = labels.reshape(-1, 1).astype(np.int16)
    ins = np.zeros_like(seg, dtype=np.int16)
    if data.shape[-1] >= 9:
        nrm = data[..., 6:9].reshape(-1, 3).astype(np.float32)
    else:
        nrm = None

    room_dir = os.path.join(OUT_ROOT, area, fname)
    os.makedirs(room_dir, exist_ok=True)
    np.save(os.path.join(room_dir, "coord.npy"), xyz)
    np.save(os.path.join(room_dir, "color.npy"), rgb)
    np.save(os.path.join(room_dir, "segment.npy"), seg)
    np.save(os.path.join(room_dir, "instance.npy"), ins)
    if nrm is not None:
        np.save(os.path.join(room_dir, "normal.npy"), nrm)
    written += 1

print(f"Done. {written} rooms written to {OUT_ROOT}/")
!ls -d ./data/s3dis/Area_* | head -10

In [ ]:
# ═══════════════════════════════════════
# CELL 3 — Train Sonata (10 epoch smoke test)
# ═══════════════════════════════════════

# Auto-select config based on masking mode
if MASKING_MODE == "cosine":
    config_file = "configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py"
elif MASKING_MODE == "vgc":
    config_file = "configs/sonata/pretrain-sonata-v1m2-vgc-smoketest.py"
else:
    config_file = "configs/sonata/pretrain-sonata-v1m2-vgc-smoketest.py"

save_path   = f"exp/{MASKING_MODE}_{GPU_COUNT}GPU_{EPOCHS}ep"
print(f"Config: {config_file}")
print(f"Masking: {MASKING_MODE} | Epochs: {EPOCHS} | GPU: {GPU_COUNT}")
# Build train command
train_cmd = (
    f"python tools/train.py "
    f"--config-file {config_file} "
    f"--options save_path={save_path} "
    f"epoch={EPOCHS} "
    f"{mode_flag} "
    f"data.train.datasets.0.data_root=data/s3dis "
    f"2>&1"
)

print(f"Command: {train_cmd}")
print(f"Output: {save_path}/train.log")

if GPU_COUNT > 1:
    !torchrun --nproc_per_node={GPU_COUNT} tools/train.py --config-file {config_file} --options save_path={save_path} epoch={EPOCHS} data.train.datasets.0.data_root=data/s3dis 2>&1
else:
    !python tools/train.py --config-file {config_file} --options save_path={save_path} epoch={EPOCHS} data.train.datasets.0.data_root=data/s3dis 2>&1

In [ ]:
# ═══════════════════════════════
# CELL 4 — Monitor Results
# ═══════════════════════════════

# TensorBoard
%load_ext tensorboard
%tensorboard --logdir exp --port 6006

# Or view training log
# !tail -50 exp/vgc_*/train.log

## Troubleshooting

**spconv installation fails**: Try `!pip install spconv-cu121` (CUDA 12.1).

**CUDA extension build fails**: Colab may have unexpected CUDA version. Run `!nvcc --version` to check, match spconv version accordingly.

**Out of memory**: Reduce batch_size in config or use 1 GPU.

**Data loader error**: Check `!ls data/s3dis/Area_1/` — should show room directories.